In [2]:
import re
import requests
import string
import numpy as np
import matplotlib.pyplot as plt
from collections import Counter, defaultdict
import tiktoken
from transformers import BertTokenizer
from urllib.parse import urlparse

# BERT vs GPT

In [3]:
starting_text = "Hello, my name is Pablo and I am a data scientist."

# GPT4 tokens:
gpt4_tokenizer = tiktoken.get_encoding("cl100k_base")
gpt4_tokens = gpt4_tokenizer.encode(starting_text)

# BERT tokens:
bert_tokenizer = BertTokenizer.from_pretrained('bert-base-uncased')
bert_tokens = bert_tokenizer.encode(starting_text)

print("Starting Text:", starting_text)
print("-"*50)
print("GPT-4 Tokens:", gpt4_tokens)
print("Decoded using GPT-4 tokenizer:", gpt4_tokenizer.decode(gpt4_tokens))
print("Decoded using BERT tokenizer:", bert_tokenizer.decode(gpt4_tokens))
print("-"*50)
print("BERT Tokens:", bert_tokens)
print("Decoded using GPT-4 tokenizer:", gpt4_tokenizer.decode(bert_tokens))
print("Decoded using BERT tokenizer:", bert_tokenizer.decode(bert_tokens))

Starting Text: Hello, my name is Pablo and I am a data scientist.
--------------------------------------------------
GPT-4 Tokens: [9906, 11, 856, 836, 374, 53863, 323, 358, 1097, 264, 828, 28568, 13]
Decoded using GPT-4 tokenizer: Hello, my name is Pablo and I am a data scientist.
Decoded using BERT tokenizer: lately [unused10] [unused851] [unused831] [unused369] [unused318] [unused353] æ [unused259] [unused823] straighten [unused12]
--------------------------------------------------
BERT Tokens: [101, 7592, 1010, 2026, 2171, 2003, 11623, 1998, 1045, 2572, 1037, 2951, 7155, 1012, 102]
Decoded using GPT-4 tokenizer: �.deleteceptionrgretoin wash(idateuilderadeODE coolinclude�
Decoded using BERT tokenizer: [CLS] hello, my name is pablo and i am a data scientist. [SEP]


In [4]:
# text -> GPT-4 tokens -> text -> BERT tokens

print("Starting Text:", starting_text)
print("-"*50)
gpt4_tokens = gpt4_tokenizer.encode(starting_text)
print("GPT-4 Tokens:", gpt4_tokens)
print("-"*50)
decoded_gpt4 = gpt4_tokenizer.decode(gpt4_tokens)
print("Decoded using GPT-4 tokenizer:", decoded_gpt4)
print("-"*50)
bert_tokens_from_gpt4 = bert_tokenizer.encode(decoded_gpt4)
print("BERT Tokens from GPT-4 decoded text:", bert_tokens_from_gpt4)
print("-"*50)
decoded_bert_from_gpt4 = bert_tokenizer.decode(bert_tokens_from_gpt4)
print("Decoded using BERT tokenizer from GPT-4 decoded text:", decoded_bert_from_gpt4)

Starting Text: Hello, my name is Pablo and I am a data scientist.
--------------------------------------------------
GPT-4 Tokens: [9906, 11, 856, 836, 374, 53863, 323, 358, 1097, 264, 828, 28568, 13]
--------------------------------------------------
Decoded using GPT-4 tokenizer: Hello, my name is Pablo and I am a data scientist.
--------------------------------------------------
BERT Tokens from GPT-4 decoded text: [101, 7592, 1010, 2026, 2171, 2003, 11623, 1998, 1045, 2572, 1037, 2951, 7155, 1012, 102]
--------------------------------------------------
Decoded using BERT tokenizer from GPT-4 decoded text: [CLS] hello, my name is pablo and i am a data scientist. [SEP]


In [5]:
print(len(gpt4_tokens), "GPT-4 tokens") 
print(len(bert_tokens), "BERT tokens")

# we can see that they are different tokenization methods, so the number of tokens can differ
# GPT4's tokenizer has a lot of subwords and BERT more word-based tokens.

13 GPT-4 tokens
15 BERT tokens


In [6]:
txt1 = "start        end"
txt2 = "start\t\t\t\t\t\t\tend"
txt3 = "start\r\n\r\n\r\n\r\n\r\n\r\nend"


bert_txt1_tokens = bert_tokenizer.encode(txt1)
bert_txt2_tokens = bert_tokenizer.encode(txt2)
bert_txt3_tokens = bert_tokenizer.encode(txt3)
print("BERT Tokens for txt1:", bert_txt1_tokens, "->", bert_tokenizer.decode(bert_txt1_tokens))
print("BERT Tokens for txt2:", bert_txt2_tokens, "->", bert_tokenizer.decode(bert_txt2_tokens))
print("BERT Tokens for txt3:", bert_txt3_tokens, "->", bert_tokenizer.decode(bert_txt3_tokens))
# for BERT, all the different whitespace characters are treated as a single space, so they all get tokenized to the same token, 
# which is why we see the same token for txt1, txt2 and txt3. This is because BERT's tokenizer is designed to be robust to different 
# types of whitespace and treats them as equivalent.
# This is because BERT's tokenizer is designed for classification tasks, therefore a whitespace is not important for the meaning of the text.

gpt4_txt1_tokens = gpt4_tokenizer.encode(txt1)
gpt4_txt2_tokens = gpt4_tokenizer.encode(txt2)
gpt4_txt3_tokens = gpt4_tokenizer.encode(txt3)
print("GPT-4 Tokens for txt1:", gpt4_txt1_tokens, "->", gpt4_tokenizer.decode(gpt4_txt1_tokens))
print("GPT-4 Tokens for txt2:", gpt4_txt2_tokens, "->", gpt4_tokenizer.decode(gpt4_txt2_tokens))
print("GPT-4 Tokens for txt3:", gpt4_txt3_tokens, "->", gpt4_tokenizer.decode(gpt4_txt3_tokens))
# for GPT-4, the different whitespace characters are treated differently, so they get tokenized to different tokens,
# which is why we see different tokens for txt1, txt2 and txt3. This is because GPT-4's tokenizer is designed to be more sensitive to
# different types of whitespace and treats them as distinct tokens.
# As GPT-4's tokenizer is designed for generation tasks, therefore a whitespace can be important for the meaning of the text, especially for formatting purposes.

BERT Tokens for txt1: [101, 2707, 2203, 102] -> [CLS] start end [SEP]
BERT Tokens for txt2: [101, 2707, 2203, 102] -> [CLS] start end [SEP]
BERT Tokens for txt3: [101, 2707, 2203, 102] -> [CLS] start end [SEP]
GPT-4 Tokens for txt1: [2527, 286, 842] -> start        end
GPT-4 Tokens for txt2: [2527, 1696, 6379] -> start							end
GPT-4 Tokens for txt3: [2527, 27333, 881, 408] -> start





end


In [7]:
baseurl = 'https://www.gutenberg.org/cache/epub/'

bookurls = [
    # code       title
    ['84',    'Frankenstein'    ],
    ['64317', 'GreatGatsby'     ],
    ['11',    'AliceWonderland' ],
    ['1513',  'RomeoJuliet'     ],
    ['76',    'HuckFinn'        ],
    ['219',   'HeartDarkness'   ],
    ['2591',  'GrimmsTales'     ],
    ['2148',  'EdgarAllenPoe'   ],
    ['36',    'WarOfTheWorlds'  ],
    ['829',   'GulliversTravels']
]

print('  Book title     |  Chars  | GPT-4 Compression | BERT Compression')
print('-'*100)

for code,title in bookurls:

  # get the text
  fullurl = baseurl + code + '/pg' + code + '.txt'
  text = requests.get(fullurl).text
  num_chars = len(text)

  # tokenize
  gpt4_tokens = gpt4_tokenizer.encode(text)
  gpt4_num_tokens = len(gpt4_tokens)

  bert_tokens = bert_tokenizer.encode(text)
  bert_num_tokens = len(bert_tokens)

  # compression ratio, the lower the better, as it means that we are using fewer tokens to represent the same text
  gpt4_compress = gpt4_num_tokens / num_chars * 100
  bert_compress = bert_num_tokens / num_chars * 100

  print(f'{title:16} | {num_chars:>7,d} |        {gpt4_compress:>3.2f}%      |       {bert_compress:>3.2f}%')

  # low variability in the compression, between 20 and 25%

  Book title     |  Chars  | GPT-4 Compression | BERT Compression
----------------------------------------------------------------------------------------------------


Token indices sequence length is longer than the specified maximum sequence length for this model (96281 > 512). Running this sequence through the model will result in indexing errors


Frankenstein     | 446,583 |        22.93%      |       21.56%
GreatGatsby      | 296,900 |        23.68%      |       23.82%
AliceWonderland  | 167,713 |        24.70%      |       24.39%
RomeoJuliet      | 167,470 |        26.12%      |       25.16%
HuckFinn         | 602,753 |        26.40%      |       26.15%
HeartDarkness    | 232,925 |        24.24%      |       23.20%
GrimmsTales      | 549,777 |        24.96%      |       23.81%
EdgarAllenPoe    | 632,177 |        22.82%      |       20.59%
WarOfTheWorlds   | 363,441 |        23.27%      |       21.84%
GulliversTravels | 611,782 |        23.46%      |       22.11%


In [8]:
weburls = [
    'http://python.org/',
    'https://pytorch.org/',
    'https://en.wikipedia.org/wiki/List_of_English_words_containing_Q_not_followed_by_U',
    'https://sudoku.com/',
    'https://reddit.com/',
    'https://visiteurope.com/en/',
    'https://sincxpress.com/',
    'https://openai.com/',
    'https://theuselessweb.com/',
    'https://maps.google.com/',
    'https://pigeonsarentreal.co.uk/',
]

print('    Website       |  Chars  | GPT-4 Compression | BERT Compression')
print('-'*100)

for url in weburls:

  # get the text
  text = requests.get(url).text
  num_chars = len(text)

  # tokenize
  gpt4_tokens = gpt4_tokenizer.encode(text)
  gpt4_num_tokens = len(gpt4_tokens)

  bert_tokens = bert_tokenizer.encode(text)
  bert_num_tokens = len(bert_tokens)

  # compression ratio
  gpt4_compress = gpt4_num_tokens / num_chars * 100
  bert_compress = bert_num_tokens / num_chars * 100

  print(f'{urlparse(url).hostname[:-4]:18} | {num_chars:>7,d} |       {gpt4_compress:>3.2f}%      |       {bert_compress:>3.2f}%')

  # we can see that generally GPT-4's tokenizer is more efficient than BERT's tokenizer, as it produces fewer tokens for the same text. 
  # This is because GPT-4's tokenizer is designed to be more compact and efficient, while BERT's tokenizer is designed to be more robust 
  # and flexible. However, the compression ratio can vary depending on the specific text being tokenized, as some texts may contain more
  # subwords or special characters that can affect the tokenization process. 

    Website       |  Chars  | GPT-4 Compression | BERT Compression
----------------------------------------------------------------------------------------------------
python             |  49,310 |       25.55%      |       34.96%
pytorch            | 194,328 |       30.74%      |       48.23%
en.wikipedia       |     126 |       26.98%      |       38.89%
sudoku             | 145,556 |       36.19%      |       44.35%
reddit             |   8,366 |       38.36%      |       47.36%
visiteurope        | 318,441 |       33.23%      |       41.93%
sincxpress         |  25,392 |       28.05%      |       34.92%
openai             |  11,860 |       55.97%      |       56.17%
theuselessweb      |   4,756 |       27.94%      |       35.60%
maps.google        |  33,322 |       32.50%      |       37.99%
pigeonsarentreal.c | 243,854 |       29.21%      |       44.45%


In [ ]:
languages = ['English','Spanish','Arabic','Persian','Lithuanian','Chinese','Tamil','Esperanto']

sentences = [ 'Blue towels are great because they remind you of the sea, although the sea is wet and towels work better when they are dry.',
              'Las toallas azules son geniales porque recuerdan al mar, aunque el mar está mojado y las toallas funcionan mejor cuando están secas.',
              'تعتبر المناشف الزرقاء رائعة لأنها تذكرك بالبحر، على الرغم من أن البحر مبلل والمناشف تعمل بشكل أفضل عندما تكون جافة.',
              'حوله‌های آبی عالی هستند زیرا شما را به یاد دریا می‌اندازند، اگرچه دریا مرطوب است و حوله‌ها وقتی خشک باشند بهتر عمل می‌کنند.',
              'Mėlyni rankšluosčiai puikūs, nes primena jūrą, nors jūra yra šlapia, o rankšluosčiai geriau tinka, kai yra sausi.',
              '蓝色毛巾很棒，因为它们会让您想起大海，尽管海水是湿的，而毛巾在干燥时效果更好。',
              'நீல நிற துண்டுகள் சிறந்தவை, ஏனென்றால் அவை கடலை நினைவூட்டுகின்றன, இருப்பினும் கடல் ஈரமாக இருக்கும், துண்டுகள் உலர்ந்திருக்கும் போது சிறப்பாக வேலை செய்யும்.',
              'Bluaj mantukoj estas bonegaj ĉar ili memorigas vin pri la maro, kvankam la maro estas malseka kaj mantukoj funkcias pli bone kiam ili estas sekaj.',
]

# table header
print(' Language  |  Chars  |  BERT  |  GPT ')
print('-'*37)

for lang,text in zip(languages,sentences):

  # tokenize the text
  tokensG = gpt4_tokenizer.encode(text)
  tokensB = bert_tokenizer.encode(text)[1:-1]

  # print the result
  print(f'{lang:>10} |   {len(text):3}   |  {len(tokensB):3}   |  {len(tokensG):3}')
  # in some languages, more than compression we can see an expanding. for example in chinese, GPT needs more tokens than original characters.
  # tokenization is less effective in languages they have less training data on.

 Language  |  Chars  |  BERT  |  GPT 
-------------------------------------
   English |   123   |   26   |   26
   Spanish |   132   |   46   |   34
    Arabic |   115   |   95   |   84
   Persian |   123   |   96   |   90
Lithuanian |   113   |   46   |   57
   Chinese |    39   |   39   |   55
     Tamil |   154   |   35   |  209
 Esperanto |   146   |   56   |   50
